# Worflow of this notebook:

## Input
- a `merge.csv` file containing spot coordinates **in pixels** and names of `.msr` images from which the spot has been derived in the format `originalname_chX.tif`


## Outpout
- the orginal `merge_global_coords.csv` file with the addition of global coordinates **in nm**, located in the same folder as the original file

# 0) Imports and functions

In [ ]:
from msr import OBFFile
import numpy as np
from xml.etree import ElementTree   
import glob
import os
import re

import pandas as pd
import numpy as np

In [ ]:
# function reads msr file and extracts information
def read_msr(file):
    
    with OBFFile(file) as f:

            imgs = []
            channel_names = []

            for idx in range(0,len(f.sizes)):

                # reading image data
                img = f.read_stack(idx) # read stack with index idx into numpy array
                imgs.append(img)

                # metadata
                stack_sizes = f.sizes # list of stack sizes/shapes, including stack and dimension names
                pixel_sizes = f.pixel_sizes # like sizes, but with pixel sizes (unit: meters)
                channel_names.append(pixel_sizes[idx].name)
                pixel_sizes = [pixel_sizes[idx].sizes['ExpControl Z'],
                               pixel_sizes[idx].sizes['ExpControl Y'],
                               pixel_sizes[idx].sizes['ExpControl X']],


                # impsector metadata
                stack_footer = f.stack_footers[idx] # get footer of stack idx
                xml_imspector_metadata = stack_footer.tag_dictionary['imspector']
                et = ElementTree.fromstring(xml_imspector_metadata)

                stage_position = [float(et.find('doc/ExpControl/scan/range/coarse_z/g_off').text) +
                                  float(et.find('doc/ExpControl/scan/range/coarse_z/off').text) +
                                  float(et.find('doc/ExpControl/scan/range/z/g_off').text) +
                                  float(et.find('doc/ExpControl/scan/range/z/off').text),

                                  float(et.find('doc/ExpControl/scan/range/coarse_y/g_off').text) +
                                  float(et.find('doc/ExpControl/scan/range/coarse_y/off').text) +
                                  float(et.find('doc/ExpControl/scan/range/y/g_off').text) +
                                  float(et.find('doc/ExpControl/scan/range/y/off').text),

                                  float(et.find('doc/ExpControl/scan/range/coarse_x/g_off').text) + 
                                  float(et.find('doc/ExpControl/scan/range/coarse_x/off').text) +
                                  float(et.find('doc/ExpControl/scan/range/x/g_off').text) +
                                  float(et.find('doc/ExpControl/scan/range/x/off').text)]

            imgs = np.asarray(imgs)
            return(imgs,pixel_sizes,stage_position)
        

# function takes spot coordinate file and raw msr location -> assings global coordinate to each spot
def get_global_coords(data,raw_folder,out_path,x_name="x",y_name="y",z_name="z"):

    data_all = []

    for img in data['img'].unique(): #get all unique image names

        # subset dataframe for that name 
        data_current = data[data['img'] == img].copy()

        # transform file name to match raw msr img
        name = os.path.basename(img)
        pattern = r'_ch\d+\.tif$'
        name1 = re.sub(pattern, '.msr', name)

        # get stage position and pixel size for each image zyx
        img_array, pixel_sizes, stage_position = read_msr(f'{raw_folder}{name1}')
        pixel_sizes =  pixel_sizes * 1 # convert to mm
        stage_position = stage_position *1 # convert to mm

        # assign global spot position

        data_current['x_global_nm'] = data_current[x_name] * pixel_sizes[0][2] * 1000000 + stage_position[2] * 1000000
        data_current['y_global_nm'] = data_current[y_name] * pixel_sizes[0][1] * 1000000 + stage_position[1] * 1000000
        data_current['z_global_nm'] = data_current[z_name] * pixel_sizes[0][0] * 1000000 + stage_position[0] * 1000000

        data_all.append(data_current)

    data_all = pd.concat(data_all)

    data_all.to_csv(f'{out_path}', index=False)

# 1) Set parameters

In [ ]:
in_path = None #path to upper level directory

# subpaths
raw_subpath = "/raw/"
spots_subpath = "/detections/merge.csv"
out_subpath = "/detections/merge_global_coords.csv"

# the naming of xyz variables for coordinates in pixels in your df
x_name = 'x'
y_name = 'y'
z_name = 'z'

In [ ]:
spots = f"{in_path}{spots_subpath}"
raw_folder = f"{in_path}/{raw_subpath}"
out_path = f"{in_path}/{out_subpath}"

# read spots
data = pd.read_csv(spots)

# 2) Run

In [ ]:
get_global_coords(data,raw_folder,out_path,x_name=x_name,y_name=y_name,z_name=z_name)